# Fine-tune SegFormer on FoodSeg103 (Kaggle version)

**Set these in the notebook's right-hand Settings panel BEFORE running:**
1. **Accelerator:** a GPU (T4 x2 or P100). This uses one GPU by default.
2. **Internet:** On (needed to download the dataset and model from Hugging Face; Kaggle may ask you to verify your phone number).
3. **Persistence:** Files only (so /kaggle/working outputs are kept).

**Best way to run the long training:** top-right **Save Version > Save & Run All (Commit)**. It runs the whole notebook in the background, you can close the tab, and the outputs (checkpoints, `final_model.zip`) are saved under the version's **Output** tab.

Untested. If a cell errors, paste the traceback back.

In [1]:
!nvidia-smi
%pip install -q -U transformers datasets accelerate

Wed Nov 11 19:12:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os
# One GPU, so the effective batch size = BS * ACC.
# To use both T4s, delete the next line and set ACC = 2 in the config cell (batch is then BS * 2 GPUs * ACC).
OUTPUT_ROOT = "/kaggle/working"

os.environ["PYTORCH_NVML_BASED_CUDA_CHECK"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
print("cuda available:", torch.cuda.is_available(), "| GPUs visible:", torch.cuda.device_count())
assert torch.cuda.is_available(), "No GPU: set Settings > Accelerator to a GPU and rerun from the top"

import socket
print(socket.gethostbyname("huggingface.co"))

cuda available: True | GPUs visible: 2
3.167.112.45


In [3]:
# ---------------- CONFIG ----------------
MODEL  = "nvidia/segformer-b5-finetuned-ade-640-640"   # or "nvidia/mit-b2" (ImageNet-only start; reached about the same accuracy in your two runs)
EPOCHS = 25      # use 2 for a smoke test
BS     = 2       # per-step batch (memory limit on a 16 GB GPU)
ACC    = 4       # gradient accumulation: effective batch = BS * ACC = 16
LR     = 6e-5
SIZE   = 512
N_VAL_DURING_TRAINING = 1000

OUT   = OUTPUT_ROOT + "/foodseg103_" + MODEL.split("/")[-1] + "_acc4"
FINAL = OUT + "/final"
print(OUT)

/kaggle/working/foodseg103_segformer-b5-finetuned-ade-640-640_acc4


In [4]:
from datasets import load_dataset
import numpy as np

ds = load_dataset("EduardoPacheco/FoodSeg103")
print(ds)

# Sanity check: label masks should be class ids 0..103 (0 = background)
ex = ds["train"][0]
lab = ex["label"]
print("label mode:", lab.mode, "| unique values:", np.unique(np.array(lab)), "| classes_on_image:", ex["classes_on_image"])
assert np.array(lab).max() <= 103, "Label values are not class ids; check the dataset format before training"

README.md:   0%|          | 0.00/6.71k [00:00<?, ?B/s]

data/train-00000-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  351MB            

data/train-00000-of-00003.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  357MB            

data/train-00001-of-00003.parquet: downloading bytes:           |  0.00B            

data/train-00002-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  431MB            

... (2 outputs left out)

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/4983 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2135 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'label', 'classes_on_image', 'id'],
        num_rows: 4983
    })
    validation: Dataset({
        features: ['image', 'label', 'classes_on_image', 'id'],
        num_rows: 2135
    })
})
label mode: L | unique values: [ 0 48 66 90] | classes_on_image: [0, 48, 66, 90]


In [5]:
# FoodSeg103 class names, in the dataset card's order (0 = background)
FOODSEG103 = ["background", "candy", "egg tart", "french fries", "chocolate", "biscuit", "popcorn",
 "pudding", "ice cream", "cheese butter", "cake", "wine", "milkshake", "coffee", "juice", "milk",
 "tea", "almond", "red beans", "cashew", "dried cranberries", "soy", "walnut", "peanut", "egg",
 "apple", "date", "apricot", "avocado", "banana", "strawberry", "cherry", "blueberry", "raspberry",
 "mango", "olives", "peach", "lemon", "pear", "fig", "pineapple", "grape", "kiwi", "melon", "orange",
 "watermelon", "steak", "pork", "chicken duck", "sausage", "fried meat", "lamb", "sauce", "crab",
 "fish", "shellfish", "shrimp", "soup", "bread", "corn", "hamburg", "pizza", "hanamaki baozi",
 "wonton dumplings", "pasta", "noodles", "rice", "pie", "tofu", "eggplant", "potato", "garlic",
 "cauliflower", "tomato", "kelp", "seaweed", "spring onion", "rape", "ginger", "okra", "lettuce",
 "pumpkin", "cucumber", "white radish", "carrot", "asparagus", "bamboo shoots", "broccoli",
 "celery stick", "cilantro mint", "snow peas", "cabbage", "bean sprouts", "onion", "pepper",
 "green beans", "french beans", "king oyster mushroom", "shiitake", "enoki mushroom",
 "oyster mushroom", "white button mushroom", "salad", "other ingredients"]
NUM = len(FOODSEG103)
assert NUM == 104
id2label = {i: n for i, n in enumerate(FOODSEG103)}
label2id = {n: i for i, n in id2label.items()}
RICE_ID = label2id["rice"]
print(NUM, "classes; rice id =", RICE_ID)

104 classes; rice id = 66


In [6]:
import random, torch
from PIL import Image
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from collections import Counter
import math

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
jitter = T.ColorJitter(0.3, 0.3, 0.3, 0.05)

def _mask(lab):
    # np.array keeps palette indices intact (convert("L") would not)
    return Image.fromarray(np.array(lab).astype(np.uint8))

def _finish(img, lab):
    x = TF.normalize(TF.to_tensor(img), MEAN, STD)
    y = torch.from_numpy(np.array(lab)).long()
    return x, y

def prep_train(img, lab):
    img = img.convert("RGB").resize((SIZE, SIZE), Image.BILINEAR)   # squash-resize, same as the HF processor at inference
    lab = _mask(lab).resize((SIZE, SIZE), Image.NEAREST)
    ns = int(SIZE * random.uniform(0.5, 2))                      # random scale
    img = img.resize((ns, ns), Image.BILINEAR)
    lab = lab.resize((ns, ns), Image.NEAREST)
    if ns < SIZE:                                                    # pad up to SIZE (255 = ignored in the loss)
        p = SIZE - ns
        img = TF.pad(img, [0, 0, p, p], fill=0)
        lab = TF.pad(lab, [0, 0, p, p], fill=255)
    W, H = img.size
    x0, y0 = random.randint(0, W - SIZE), random.randint(0, H - SIZE)
    img = img.crop((x0, y0, x0 + SIZE, y0 + SIZE))
    lab = lab.crop((x0, y0, x0 + SIZE, y0 + SIZE))
    if random.random() < 0.5:
        img, lab = TF.hflip(img), TF.hflip(lab)
    return _finish(jitter(img), lab)

def prep_eval(img, lab):
    img = img.convert("RGB").resize((SIZE, SIZE), Image.BILINEAR)
    lab = _mask(lab).resize((SIZE, SIZE), Image.NEAREST)
    return _finish(img, lab)

def make_tf(fn):
    def f(batch):
        xs, ys = zip(*[fn(i, l) for i, l in zip(batch["image"], batch["label"])])
        return {"pixel_values": list(xs), "labels": list(ys)}
    return f

base = ds["train"]
cls_lists = base.data.column("classes_on_image").to_pylist()   # reads the raw table, bypasses the transform
n_img = len(cls_lists)
freq = Counter(c for cs in cls_lists for c in set(cs))

T = 0.03   # classes in fewer than 10% of images get oversampled
def repeats(cs):
    r = max([max(1.0, math.sqrt(T / (freq[c] / n_img))) for c in set(cs) if c != 0] or [1.0])
    return int(r) + (random.random() < r - int(r))

random.seed(0)
idx = [i for i, cs in enumerate(cls_lists) for _ in range(repeats(cs))]
print(n_img, "->", len(idx), "images per epoch")

train_ds = base.select(idx)
train_ds.set_transform(make_tf(prep_train))

val_full = ds["validation"];  val_full.set_transform(make_tf(prep_eval))
val_small = ds["validation"].select(range(N_VAL_DURING_TRAINING)); val_small.set_transform(make_tf(prep_eval))
print(n_img, "->", len(idx), "images per epoch")
print(len(train_ds), len(val_full), len(val_small))

4983 -> 6627 images per epoch
4983 -> 6627 images per epoch
6627 2135 1000


In [7]:
from transformers import SegformerForSemanticSegmentation, Trainer, TrainingArguments, SegformerImageProcessor

model = SegformerForSemanticSegmentation.from_pretrained(
    MODEL, num_labels=NUM, id2label=id2label, label2id=label2id, ignore_mismatched_sizes=True)
print("params:", sum(p.numel() for p in model.parameters()) / 1e6, "M")

def confusion(preds, labels):
    m = (labels != 255) & (labels >= 0)
    return np.bincount(labels[m].astype(np.int64) * NUM + preds[m].astype(np.int64),
                       minlength=NUM * NUM).reshape(NUM, NUM)

def iou_from_conf(conf):
    tp = np.diag(conf).astype(float); gt = conf.sum(1); pr = conf.sum(0)
    union = gt + pr - tp
    iou = np.where(union > 0, tp / np.maximum(union, 1), np.nan)
    present = gt > 0
    return iou, float(np.nanmean(iou[present])), float(tp.sum() / max(conf.sum(), 1))

def preprocess_logits_for_metrics(logits, labels):
    if isinstance(logits, tuple):
        logits = logits[0]
    up = torch.nn.functional.interpolate(logits, size=labels.shape[-2:], mode="bilinear", align_corners=False)
    return up.argmax(dim=1).to(torch.uint8)       # keeps eval memory small

def compute_metrics(p):
    iou, miou, acc = iou_from_conf(confusion(p.predictions, p.label_ids))
    return {"mean_iou": miou, "pixel_acc": acc, "rice_iou": float(iou[RICE_ID]) if not np.isnan(iou[RICE_ID]) else 0.0}

import math, inspect

# make sure the saved config carries the FoodSeg103 names, not the 1000 ImageNet ones
model.config.id2label = id2label
model.config.label2id = label2id
print(model.decode_head.classifier)   # should show out_channels=104

total_steps = math.ceil(len(train_ds) / (BS * ACC)) * EPOCHS
kw = dict(
    output_dir=OUT, learning_rate=LR, num_train_epochs=EPOCHS,
    per_device_train_batch_size=BS, per_device_eval_batch_size=BS,
    gradient_accumulation_steps=ACC,
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model="mean_iou", greater_is_better=True,
    logging_steps=50, remove_unused_columns=False, fp16=True,
    dataloader_num_workers=2, warmup_steps=int(0.05 * total_steps), weight_decay=0.01,
    eval_accumulation_steps=4, report_to="none")

ok = set(inspect.signature(TrainingArguments.__init__).parameters)
if "eval_strategy" not in ok and "evaluation_strategy" in ok:   # older transformers versions
    kw["evaluation_strategy"] = kw.pop("eval_strategy")
dropped = [k for k in kw if k not in ok]
if dropped:
    print("WARNING, not supported by this transformers version and dropped:", dropped)

config.json:   0%|          | 0.00/6.89k [00:00<?, ?B/s]

[transformers] You passed `num_labels=104` which is incompatible to the `id2label` map of length `150`.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  339MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  339MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1172 [00:00<?, ?it/s]

[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/segformer-b5-finetuned-ade-640-640
Key                           | Status   |                                                                                                       
------------------------------+----------+-------------------------------------------------------------------------------------------------------
decode_head.classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150]) vs model:torch.Size([104])                      
decode_head.classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150, 768, 1, 1]) vs model:torch.Size([104, 768, 1, 1])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


params: 84.67332 M
Conv2d(768, 104, kernel_size=(1, 1), stride=(1, 1))


In [ ]:
import glob
from accelerate import notebook_launcher

resume = len(glob.glob(OUT + "/checkpoint-*")) > 0

print("CUDA initialized in main process:", torch.cuda.is_initialized())

class HeadLRTrainer(Trainer):
    def create_optimizer(self):
        if self.optimizer is None:
            head, body = [], []
            for n, p in self.model.named_parameters():
                if p.requires_grad:
                    (head if "decode_head" in n else body).append(p)
            lr = self.args.learning_rate
            self.optimizer = torch.optim.AdamW(
                [{"params": body, "lr": lr}, {"params": head, "lr": lr * 10}],
                weight_decay=self.args.weight_decay)
        return self.optimizer

def train_fn():
    m = SegformerForSemanticSegmentation.from_pretrained(
        MODEL, num_labels=NUM, id2label=id2label, label2id=label2id,
        ignore_mismatched_sizes=True)
    m.config.id2label, m.config.label2id = id2label, label2id
    args = TrainingArguments(**{k: v for k, v in kw.items() if k in ok})
    tr = HeadLRTrainer(model=m, args=args, train_dataset=train_ds, eval_dataset=val_small,
                 compute_metrics=compute_metrics,
                 preprocess_logits_for_metrics=preprocess_logits_for_metrics)
    tr.train(resume_from_checkpoint=resume if resume else None)

notebook_launcher(train_fn, num_processes=2)

CUDA initialized in main process: False
Launching training on 2 CUDAs.


[transformers] You passed `num_labels=104` which is incompatible to the `id2label` map of length `150`.
[transformers] You passed `num_labels=104` which is incompatible to the `id2label` map of length `150`.


Loading weights:   0%|          | 0/1172 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1172 [00:00<?, ?it/s]

[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/segformer-b5-finetuned-ade-640-640
Key                           | Status   |                                                                                                       
------------------------------+----------+-------------------------------------------------------------------------------------------------------
decode_head.classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150]) vs model:torch.Size([104])                      
decode_head.classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150, 768, 1, 1]) vs model:torch.Size([104, 768, 1, 1])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.
[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/segformer-b5-finetuned-ade-640-640
Key                           | Status   |                                                       

Epoch,Training Loss,Validation Loss,Mean Iou,Pixel Acc,Rice Iou
1,16.925500,2.727102,0.090550,0.666283,0.338854
2,11.405854,1.867672,0.214077,0.742992,0.649342
3,8.665074,1.719647,0.286897,0.757551,0.635391
4,7.466395,1.594839,0.336077,0.776242,0.699349
5,6.335127,1.654996,0.346392,0.775923,0.715083
6,5.566916,1.646196,0.359537,0.780616,0.644184
7,5.043237,1.644171,0.384497,0.787294,0.726871
8,4.471879,1.726455,0.370590,0.782500,0.666856
9,3.571135,1.785821,0.388866,0.784696,0.746046
10,3.664870,1.800389,0.386919,0.789159,0.739384


... (37 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)


In [ ]:
# Save the final model from the BEST checkpoint, loaded fresh
# (Trainer's own "load best model" step can mis-load weights on newer transformers, so don't use trainer.model).
import json, glob
ckpts = sorted(glob.glob(OUT + "/checkpoint-*"), key=lambda p: int(p.split("-")[-1]))
state = json.load(open(ckpts[-1] + "/trainer_state.json"))
BEST = state.get("best_model_checkpoint") or ckpts[-1]
print("best checkpoint:", BEST)

best_model = SegformerForSemanticSegmentation.from_pretrained(BEST)
best_model.save_pretrained(FINAL)
SegformerImageProcessor(size={"height": SIZE, "width": SIZE}, do_reduce_labels=False).save_pretrained(FINAL)

import shutil
shutil.make_archive("/kaggle/working/final_model", "zip", FINAL)
print("saved to", FINAL, "and /kaggle/working/final_model.zip (download it from the Output tab)")

In [ ]:
# Full validation set: overall mIoU, rice IoU, and the weakest classes
from torch.utils.data import DataLoader
from transformers import SegformerForSemanticSegmentation

model = SegformerForSemanticSegmentation.from_pretrained(FINAL).cuda().eval()   # evaluate the saved model, freshly loaded
conf = np.zeros((NUM, NUM), dtype=np.int64)
loader = DataLoader(val_full, batch_size=BS, num_workers=2)

def tta_probs(model, x, scales=(0.75, 1.0, 1.25)):
    probs = 0
    for s in scales:
        xs = x if s == 1.0 else torch.nn.functional.interpolate(
            x, scale_factor=s, mode="bilinear", align_corners=False)
        for flip in (False, True):
            xi = xs.flip(-1) if flip else xs
            lg = model(pixel_values=xi).logits
            lg = torch.nn.functional.interpolate(
                lg.float(), size=(SIZE, SIZE), mode="bilinear", align_corners=False)
            if flip:
                lg = lg.flip(-1)
            probs = probs + lg.softmax(1)
    return probs

with torch.no_grad():
    for b in loader:
        with torch.autocast("cuda", dtype=torch.float16):
            pr = tta_probs(model, b["pixel_values"].cuda())
        conf += confusion(pr.argmax(1).cpu().numpy(), b["labels"].numpy())
iou, miou, acc = iou_from_conf(conf)
print(f"mIoU {miou:.3f} | pixel acc {acc:.3f} | rice IoU {iou[RICE_ID]:.3f}")
order = np.argsort(np.nan_to_num(iou, nan=9))
print("weakest classes:", [(id2label[i], round(float(iou[i]), 2)) for i in order[:12] if not np.isnan(iou[i])])

In [ ]:
c = conf
union = c.sum(1) + c.sum(0) - np.diag(c)
iou_d = np.where(union > 0, np.diag(c) / np.maximum(union, 1), np.nan)

for i in np.where(np.nan_to_num(iou_d, nan=9) < 0.05)[0]:
    gt, pr = c[i].sum(), c[:, i].sum()
    top = [j for j in np.argsort(-c[i])[:3] if j != i][:2]
    mostly = ", ".join(f"{id2label[j]} {c[i, j] / max(gt, 1):.0%}" for j in top)
    print(f"{id2label[i]:<22} gt_px={gt:>9} pred_px={pr:>9}  mostly predicted as: {mostly}")

In [ ]:
import glob
print(glob.glob("/kaggle/working/*"))
print(glob.glob("/kaggle/working/*/final"))

In [ ]:
# Try it on your own photo
import matplotlib.pyplot as plt
from PIL import Image
# Add your photo via "Add data > Upload" on the right, then set the path below.
PHOTO = "/kaggle/input/YOUR-DATASET-NAME/yumgrub.jpg"   # <- edit this
img = Image.open(PHOTO).convert("RGB")

proc = SegformerImageProcessor.from_pretrained(FINAL)
inp = proc(images=img, return_tensors="pt").to("cuda")
with torch.no_grad():
    lg = model(**inp).logits
lg = torch.nn.functional.interpolate(lg, size=img.size[::-1], mode="bilinear", align_corners=False)
cm = lg.argmax(1)[0].cpu().numpy()

for i, c in sorted(zip(*np.unique(cm, return_counts=True)), key=lambda x: -x[1]):
    if c / cm.size > 0.005:
        print(f"{i:>4} {id2label[int(i)]:<24} {c / cm.size:.3f}")

pal = np.random.RandomState(0).randint(30, 255, (NUM, 3)).astype(np.uint8); pal[0] = 0
blend = np.where((cm > 0)[..., None], 0.5 * np.asarray(img) + 0.5 * pal[cm], np.asarray(img)).astype(np.uint8)
plt.figure(figsize=(12, 6)); plt.subplot(1, 2, 1); plt.imshow(img); plt.axis("off")
plt.subplot(1, 2, 2); plt.imshow(blend); plt.axis("off"); plt.show()